In [ ]:
## SIDEWALKS

In [32]:
import geopandas as gpd

url = (
    "https://arcgis.dvrpc.org/portal/rest/services/transportation/"
    "PedestrianNetwork_lines/FeatureServer/0/query?"
    "where=1%3D1&outFields=*&f=geojson"
)

ped = gpd.read_file(url)


KeyboardInterrupt: 

In [41]:
ped.columns

Index(['objectid', 'line_type', 'material', 'material_o', 'feat_type',
       'raised', 'width', 'captured', 'state', 'county', 'muni', 'community',
       'ped_sig', 'created_user', 'created_date', 'last_edited_user',
       'last_edited_date', 'vertchange', 'cracking', 'cross_slope',
       'fix_obstr', 'veg', 'over_obj', 'buff_zone', 'st_light', 'tree',
       'cond_date', 'notes', 'globalid', 'Shape__Length', 'geometry',
       'length_m'],
      dtype='object')

In [84]:
#THIS WILL BE USED AT THE END
#I WANT TO CALCULATE INDEX FOR WHOLE CITY FIRST
import geopandas as gpd
tracts = gpd.read_file("philly_neighborhoods.geojson")
# Ensure CRS matches
tracts = tracts.to_crs("EPSG:3857")   
eptracts = tracts[tracts["Name"].str.contains("PASSYUNK_SQUARE", case=False, na=False)]

ped_ep= gpd.sjoin(ped, eptracts, how="inner", predicate="within")


In [137]:

# Example counts (adjust field names to match your data)
# Count sidewalk gaps
if "SIDEWALK" in ped.columns:
    sidewalk_gaps = ped["SIDEWALK"].value_counts()
    print("Sidewalk counts / gaps:\n", sidewalk_gaps)

# Count ADA ramps
if "ADA_RAMP" in ped.columns:
    ada_ramps = ped["ADA_RAMP"].value_counts()
    print("ADA ramps counts:\n", ada_ramps)

# Count crosswalks
if "CROSSWALK" in ped.columns:
    crosswalks = ped["CROSSWALK"].value_counts()
    print("Crosswalk counts:\n", crosswalks)

# Optional: summarize total counts
print("Total sidewalk segments:", len(ped))
if "ADA_RAMP" in ped.columns:
    print("Total ADA ramps:", ped_ep["ADA_RAMP"].sum())
if "CROSSWALK" in ped.columns:
    print("Total crosswalks:", ped_ep["CROSSWALK"].sum())

Total sidewalk segments: 2000


In [81]:
tracts

,Name,geometry,area_m2
0,LAWNDALE,"POLYGON ((-4600919.344 6582989.179, -4601761.4...","6,319,723.76"
1,ASTON_WOODBRIDGE,"POLYGON ((-4594640.238 6576746.557, -4594652.1...","2,356,536.23"
2,CARROLL_PARK,"POLYGON ((-4619675.091 6587784.151, -4619514.8...","2,183,361.22"
3,CHESTNUT_HILL,"POLYGON ((-4606489.428 6597348.801, -4606493.4...","13,227,276.58"
4,BURNHOLME,"POLYGON ((-4598988.554 6584941.327, -4598986.9...","1,747,195.58"
...,...,...,...
153,DICKINSON_NARROWS,"POLYGON ((-4619584.453 6576492.289, -4619901.1...","812,229.25"
154,GARDEN_COURT,"POLYGON ((-4621395.992 6584331.69, -4621718.80...","530,451.39"
155,WISSAHICKON_HILLS,"POLYGON ((-4612464.629 6593168.403, -4612535.5...","505,594.42"
156,DEARNLEY_PARK,"POLYGON ((-4614165.61 6596428.895, -4613950.19...","2,689,595.10"


In [102]:
import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)
tracts = tracts.to_crs("EPSG:32618")   
tracts['area_m2'] = tracts.geometry.area
tracts

,Name,geometry,area_m2
0,LAWNDALE,"POLYGON ((492650.696 4433325.065, 492414.12 44...","3,537,467.57"
1,ASTON_WOODBRIDGE,"POLYGON ((499266.779 4433715.944, 499265.975 4...","1,321,478.09"
2,CARROLL_PARK,"POLYGON ((480639.751 4425251.523, 481194.846 4...","1,217,644.48"
3,CHESTNUT_HILL,"POLYGON ((481860.144 4437364.762, 481864.814 4...","7,396,101.84"
4,BURNHOLME,"POLYGON ((492523.34 4435376.009, 492531.278 44...","978,620.44"
...,...,...,...
153,DICKINSON_NARROWS,"POLYGON ((486974.299 4419683.824, 486930.502 4...","453,072.28"
154,GARDEN_COURT,"POLYGON ((481707.565 4422575.867, 481660.947 4...","295,741.10"
155,WISSAHICKON_HILLS,"POLYGON ((481222.501 4431949.862, 481176.751 4...","282,463.83"
156,DEARNLEY_PARK,"POLYGON ((478557.527 4432621.566, 478665.632 4...","1,502,227.41"


In [103]:
total_area_m2 = tracts['area_m2'].sum()
print(f"Total area: {total_area_m2:,.2f} m²")


Total area: 369,285,108.40 m²


In [104]:
# Ensure the CRS is in meters for length calculation
ped = ped.to_crs(epsg=32618)
ped["length_m"] = ped.geometry.length

total_length = ped["length_m"].sum()
print("Total pedestrian network length (meters):", total_length)


Total pedestrian network length (meters): 122281.51177612302


In [109]:
tracts = tracts.to_crs(epsg=32618)
ped = ped.to_crs(epsg=32618)

sidewalk_by_neighborhood = gpd.sjoin(tracts, ped, how='left', predicate='intersects')

# Aggregate total sidewalk length per A2 neighborhood
sidewalk_summary = sidewalk_by_neighborhood.groupby('Name').agg({
    'geometry': 'first',    # keep the tract geometry
    'length_m': 'sum'       # total sidewalk length in meters
}).reset_index()

# Convert to GeoDataFrame and set geometry
sidewalk_summary = gpd.GeoDataFrame(sidewalk_summary, geometry='geometry', crs=tracts.crs)

# Compute tract area in m²
sidewalk_summary['area_m2'] = tracts['area_m2']

# Compute sidewalk density as proportion of tract area
sidewalk_summary['sidewalk_density'] = sidewalk_summary['length_m'] / sidewalk_summary['area_m2'] * 1e6

# Inspect results
print(sidewalk_summary[['Name', 'length_m', 'area_m2', 'sidewalk_density']])

                   Name  length_m      area_m2  sidewalk_density
0       ACADEMY_GARDENS     18.97 3,537,467.57              5.36
1               AIRPORT      0.00 1,321,478.09              0.00
2        ALLEGHENY_WEST      0.00 1,217,644.48              0.00
3               ANDORRA      0.00 7,396,101.84              0.00
4      ASTON_WOODBRIDGE      0.00   978,620.44              0.00
..                  ...       ...          ...               ...
153              WISTER      7.31   453,072.28             16.13
154    WOODLAND_TERRACE     65.67   295,741.10            222.04
155          WYNNEFIELD      2.20   282,463.83              7.80
156  WYNNEFIELD_HEIGHTS    167.83 1,502,227.41            111.72
157            YORKTOWN      0.00   633,652.68              0.00

[158 rows x 4 columns]


In [116]:
# Normalize sidewalk_density to 0-1 scale
max_density = sidewalk_summary['sidewalk_density'].max()
sidewalk_summary['sidewalk_index'] = sidewalk_summary['sidewalk_density'] / max_density

# Inspect results
print(sidewalk_summary[['Name', 'sidewalk_density', 'sidewalk_index']])
sidewalk_summary

                   Name  sidewalk_density  sidewalk_index
0       ACADEMY_GARDENS              5.36            0.00
1               AIRPORT              0.00            0.00
2        ALLEGHENY_WEST              0.00            0.00
3               ANDORRA              0.00            0.00
4      ASTON_WOODBRIDGE              0.00            0.00
..                  ...               ...             ...
153              WISTER             16.13            0.00
154    WOODLAND_TERRACE            222.04            0.02
155          WYNNEFIELD              7.80            0.00
156  WYNNEFIELD_HEIGHTS            111.72            0.01
157            YORKTOWN              0.00            0.00

[158 rows x 3 columns]


,Name,geometry,length_m,area_m2,sidewalk_density,sidewalk_index
0,ACADEMY_GARDENS,"POLYGON ((500127.271 4434899.076, 500464.133 4...",18.97,"3,537,467.57",5.36,0.00
1,AIRPORT,"POLYGON ((483133.506 4415847.15, 483228.716 44...",0.00,"1,321,478.09",0.00,0.00
2,ALLEGHENY_WEST,"POLYGON ((485837.663 4428133.6, 485834.655 442...",0.00,"1,217,644.48",0.00,0.00
3,ANDORRA,"POLYGON ((480844.652 4435202.29, 480737.832 44...",0.00,"7,396,101.84",0.00,0.00
4,ASTON_WOODBRIDGE,"POLYGON ((499266.779 4433715.944, 499265.975 4...",0.00,"978,620.44",0.00,0.00
...,...,...,...,...,...,...
153,WISTER,"POLYGON ((485318.173 4432033.915, 485329.046 4...",7.31,"453,072.28",16.13,0.00
154,WOODLAND_TERRACE,"POLYGON ((482574.267 4422191.877, 482576.988 4...",65.67,"295,741.10",222.04,0.02
155,WYNNEFIELD,"POLYGON ((481162.526 4428216.067, 481397.143 4...",2.20,"282,463.83",7.80,0.00
156,WYNNEFIELD_HEIGHTS,"POLYGON ((482840.825 4428241.117, 482577.876 4...",167.83,"1,502,227.41",111.72,0.01


In [139]:
sidewalk_gaps_df['total_segments'].sum()


np.int64(232)

In [196]:

#import geopandas as gpd
#import networkx as nx
#import pandas as pd
#from shapely.geometry import Point
#from shapely.ops import snap, unary_union

# Ensure projected CRS for accurate distance/area
tracts = tracts.to_crs(epsg=2272)
ped = ped.to_crs(epsg=2272)

# Spatial join: assign sidewalks to neighborhoods
ped_by_tract = gpd.sjoin(ped, tracts[['Name', 'geometry']], how='left', predicate='intersects')

# Count total sidewalks per neighborhood
sidewalk_counts = ped_by_tract.groupby('Name').size().reset_index(name='total_sidewalks')

# Prepare list to store gap info
tract_gaps = []

tolerance = 1.0  # meters, adjust for snapping endpoints

for name, group in ped_by_tract.groupby('Name'):
    G = nx.Graph()
    
    # Create nodes for each segment's endpoints
    nodes = []
    for geom in group.geometry:
        if geom is None:
            continue
        coords = list(geom.coords)
        nodes.append(coords[0])
        nodes.append(coords[-1])
        for start, end in zip(coords[:-1], coords[1:]):
            G.add_edge(start, end)
    

    
    # Count connected components
    num_components = nx.number_connected_components(G)
    
    # True sidewalk gaps = components - 1
    num_gaps = max(num_components - 1, 0)
    
    tract_gaps.append({
        'Name': name,
        'sidewalk_gaps': num_gaps
    })

# Convert to DataFrame
gaps_df = pd.DataFrame(tract_gaps)

# Merge total sidewalks and gaps, include all neighborhoods
sidewalk_summary = tracts[['Name']].merge(sidewalk_counts, on='Name', how='left')
sidewalk_summary = sidewalk_summary.merge(gaps_df, on='Name', how='left')

# Fill NaN for neighborhoods with 0 sidewalks
sidewalk_summary['total_sidewalks'] = sidewalk_summary['total_sidewalks'].fillna(0)
sidewalk_summary['sidewalk_gaps'] = sidewalk_summary['sidewalk_gaps'].fillna(0)

# Compute gap density: proportion of gaps per total sidewalks
# Avoid division by zero
sidewalk_summary['gap_density'] = sidewalk_summary.apply(
    lambda row: row['sidewalk_gaps'] / row['total_sidewalks'] if row['total_sidewalks'] > 0 else 0,
    axis=1
)

# Compute connectivity index (1 = fully connected, 0 = fully disconnected)
sidewalk_summary['connectivity_index'] = 1 - sidewalk_summary['gap_density']

# Inspect top/bottom
print(sidewalk_summary[['Name', 'total_sidewalks', 'sidewalk_gaps', 'gap_density', 'connectivity_index']].sort_values('connectivity_index'))


                  Name  total_sidewalks  sidewalk_gaps  gap_density  \
69     UNIVERSITY_CITY            22.00          21.00         0.95   
66          CEDAR_PARK            11.00          10.00         0.91   
127  WASHINGTON_SQUARE             9.00           8.00         0.89   
108   WOODLAND_TERRACE             5.00           4.00         0.80   
150        RITTENHOUSE             5.00           4.00         0.80   
..                 ...              ...            ...          ...   
153  DICKINSON_NARROWS             0.00           0.00         0.00   
154       GARDEN_COURT             0.00           0.00         0.00   
155  WISSAHICKON_HILLS             0.00           0.00         0.00   
156      DEARNLEY_PARK             0.00           0.00         0.00   
157       GERMANY_HILL             0.00           0.00         0.00   

     connectivity_index  
69                 0.05  
66                 0.09  
127                0.11  
108                0.20  
150              

In [ ]:
##

In [202]:
import geopandas as gpd

curb_ramps = gpd.read_file(
    "https://opendata.arcgis.com/datasets/31c32aa95a2f44dba411af1f8f7f1362_0.geojson"
).to_crs(epsg=3857)



HTTPError: HTTP Error 400: Bad Request